In [58]:
from IPython.core import display_functions
from IPython.core import display_functions
from sentence_transformers.sparse_encoder.losses import SparseMultipleNegativesRankingLoss
import pandas as pd
import requests
from ragas import EvaluationDataset
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics.collections import faithfulness, answer_correctness
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os
from ragas.run_config import RunConfig
from ragas.llms import llm_factory

from langchain_google_genai import ChatGoogleGenerativeAI
from datasets import Dataset 
from langchain_community.embeddings import SentenceTransformerEmbeddings


In [2]:
load_dotenv()

df = pd.read_csv("testset.csv")
user_queries = df["user_input"]
expected_responses = df["reference"]

dataset = []

for query,reference in zip(user_queries,expected_responses):

    response = requests.post("http://localhost:8000/evaluation_query", json={"prompt": query, "history": []})
    response_json = response.json()

    relevant_docs = [doc["page_content"] for doc in response_json["source_chunks"]]
    response = response_json["answer"]
    dataset.append(
        {
            "user_input":query,
            "retrieved_contexts":relevant_docs,
            "response":response,
            "reference":reference
        }
    )

In [25]:
#setting up custom dataset
temp_list = []
temp_list1 = []


for obj in dataset:
    temp_list.append(obj['response'])
    temp_list1.append(obj['retrieved_contexts'])




In [24]:
data_samples = {
    'question':user_queries.tolist(),
    'answer':[],
    'contexts':[],
    'ground_truth':[]
}

In [26]:
data_samples['answer'] = temp_list

In [27]:
data_samples['contexts'] = temp_list1

In [28]:
data_samples['ground_truth'] = expected_responses.tolist()

In [33]:
my_dataset = Dataset.from_dict(data_samples)

In [ ]:
evaluator_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("GEMINI_API_KEY"),
    temperature=0
)

/var/folders/rg/bf6jgyyx59q1398512l2gxv40000gn/T/ipykernel_44013/686995221.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(ChatGoogleGenerativeAI(


In [56]:
faithfulness_metric = Faithfulness(llm=evaluator_llm)


score = evaluate(
    dataset=my_dataset,llm=evaluator_llm,
    metrics=[faithfulness],emebeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-V2")
    
)

ValueError: Collections metrics only support modern InstructorLLM. Found: LangchainLLMWrapper. Use: llm_factory('gpt-4o-mini', client=openai_client)

In [6]:

evaluation_dataset = EvaluationDataset.from_list(dataset)

/var/folders/rg/bf6jgyyx59q1398512l2gxv40000gn/T/ipykernel_44013/686995221.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(ChatGoogleGenerativeAI(


In [10]:
print(evaluation_dataset)

EvaluationDataset(features=['user_input', 'retrieved_contexts', 'response', 'reference'], len=21)


In [ ]:
result = evaluate(dataset=evaluation_dataset,metrics=[LLMContextRecall(), Faithfulness()],llm=evaluator_llm,run_config=RunConfig(
        max_workers=1,   # 👈 one request at a time
        timeout=180,
        max_retries=10,
        max_wait = 30
    ))
print(result)

Evaluating:  14%|█▍        | 6/42 [19:21<1:56:06, 193.52s/it]


KeyboardInterrupt: 

Exception raised in Job[6]: TimeoutError()
Exception raised in Job[7]: AssertionError(LLM is not set)
Exception raised in Job[8]: AssertionError(set LLM before use)
Exception raised in Job[9]: AssertionError(LLM is not set)
Exception raised in Job[10]: AssertionError(set LLM before use)
Exception raised in Job[11]: AssertionError(LLM is not set)
Exception raised in Job[12]: AssertionError(set LLM before use)
Exception raised in Job[13]: AssertionError(LLM is not set)
Exception raised in Job[14]: AssertionError(set LLM before use)
Exception raised in Job[15]: AssertionError(LLM is not set)
Exception raised in Job[16]: AssertionError(set LLM before use)
Exception raised in Job[17]: AssertionError(LLM is not set)
Exception raised in Job[18]: AssertionError(set LLM before use)
Exception raised in Job[19]: AssertionError(LLM is not set)
Exception raised in Job[20]: AssertionError(set LLM before use)
Exception raised in Job[21]: AssertionError(LLM is not set)
Exception raised in Job[22]: Ass